In [1]:
!sudo apt update
!sudo apt install ffmpeg

!pip install whisper-openai
!pip install InaSpeechSegmenter
!pip install pyannote.audio
!pip install --upgrade transformers

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:3 https://apt.postgresql.org/pub/repos/apt noble-pgdg InRelease            
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease               
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:7 https://ppa.launchpadcontent.net/git-core/ppa/ubuntu noble InRelease     
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
76 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: https://apt.postgresql.org/pub/repos/apt/dists/noble-pgdg/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Ke

In [16]:
import pandas as pd

s3_folder = "s3://lab/PESSD/wav"

wav_files = fs.glob(f"{s3_folder}/*.wav")
wav_files

for file in wav_files:
    fs.get(file, "data/"+(file.removeprefix("lab/PESSD/wav/")))

[]

### A ne run qu'une seule fois !

In [17]:
import whisper
import torch

torch.cuda.empty_cache()

whisper_model = whisper.load_model("medium")

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████████████████████████████████| 1.42G/1.42G [00:14<00:00, 105MiB/s]


In [ ]:

file_name="BI"
token = ""

# Code final

In [ ]:
from pyannote.audio import Pipeline
from inaSpeechSegmenter import Segmenter
from inaSpeechSegmenter.export_funcs import seg2csv
import pandas as pd
import os
import whisper

# Pipeline pour diarization
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=token
)

# Modèle InaSpeechSegmenter (genre)
seg_model = Segmenter()

# Liste des fichiers audio
audios = ["data/"+file_name+"_wav.wav"]  

#### BOUCLE PRINCIPALE
for audio_path in audios:

    # --- RESET DES LISTES / DATAFRAMES pour ce fichier ---
    diarization_list = []
    merged_list = []
    merged_list_g = []
    df_segments = pd.DataFrame(columns=["numero_segment", "start", "end", "text", "audio_file"])

    # --- TRANSCRIPTION ---
    audio = whisper.load_audio(audio_path)

    # Transcrire avec segmentation
    result = whisper_model.transcribe(audio_path, language="fr")  # force français

    # Créer une liste de dict pour chaque segment
    segments_data = [
        {
            "start": seg["start"],
            "stop": seg["end"],
            "text": seg["text"].strip(),
            "audio_file": os.path.basename(audio_path)
        }
    for seg in result["segments"]
    ]

    # Transformer en DataFrame
    df_segments = pd.DataFrame(segments_data)

    # --- DIARIZATION ---
    output = pipeline(audio_path)

    for turn, speaker in output.speaker_diarization:
        diarization_list.append({
            "speaker": str(speaker),
            "start": turn.start,
            "end": turn.end
        })

    # Fusionner les segments consécutifs du même locuteur
    for seg_item in diarization_list:
        if not merged_list:
            merged_list.append(seg_item)
        else:
            last = merged_list[-1]
            if seg_item["speaker"] == last["speaker"]:
                last["end"] = seg_item["end"]
            else:
                merged_list.append(seg_item)
    
    ### MAIN SPEAKER
    # Détecter tous les locuteurs
    speakers = sorted({s["speaker"] for s in merged_list})

    # Ajouter colonnes pour chaque speaker
    for spk in speakers:
        df_segments[spk] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_speaker"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

    # dictionnaire temporaire pour stocker les durées
        speaker_times = {spk: 0.0 for spk in speakers}

        for s in merged_list:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["end"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                speaker_times[s["speaker"]] += duration

    # remplir les colonnes du DataFrame
        for spk, t in speaker_times.items():
            df_segments.at[idx, spk] = t

    # déterminer le locuteur principal
        main_spk = max(speaker_times, key=speaker_times.get)
        df_segments.at[idx, "main_speaker"] = main_spk


    # --- GENRE LOCUTEUR ---
    segmentation = seg_model(audio_path)

    for gender, start, stop in segmentation:
    
        if not merged_list_g:
            merged_list_g.append({"gender": gender, "start": start, "stop": stop})
    
        else:
            last = merged_list_g[-1]
        
            if gender == last["gender"]:
                last["stop"] = stop
        
            else:
                merged_list_g.append({"gender": gender, "start": start, "stop": stop})

    # Détecter tous les locuteurs
    gender = sorted({s["gender"] for s in merged_list_g})

    # Ajouter colonnes pour chaque speaker
    for g in gender:
        df_segments[g] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_g"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

        # dictionnaire temporaire pour stocker les durées
        gender_times = {g: 0.0 for g in gender}

        for s in merged_list_g:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["stop"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                gender_times[s["gender"]] += duration

        # remplir les colonnes du DataFrame
        for g, t in gender_times.items():
            df_segments.at[idx, g] = t

        # déterminer le locuteur principal
        main_g = max(gender_times, key=gender_times.get)
        df_segments.at[idx, "main_g"] = main_g

    # --- METTRE LE DF AU PROPRE ---
    # Fusionner les lignes d'un même locuteur
    df_segments['bloc'] = (df_segments['main_speaker'] != df_segments['main_speaker'].shift()).cumsum()

    df_segments = df_segments.groupby('bloc').agg(
        start=('start', 'first'),
        stop=('stop', 'last'),
        text=('text', ' '.join),
        main_speaker=('main_speaker', 'first'),
        main_g=('main_g', 'first'),
        audio_file=('audio_file', 'first')

    ).reset_index(drop=True)

    # --- ENREGISTRER LE RESULTAT EN CSV POUR CE FICHIER ---
    csv_name = os.path.splitext(os.path.basename(audio_path))[0] + "_transcription.csv"
    df_segments.to_csv(csv_name, index=False, encoding="utf-8-sig")
    fs.put(csv_name, f"s3://lab/PESSD/csv/{file_name}.csv")
    print(f"CSV créé pour {audio_path} : {csv_name}")

/opt/python/lib/python3.13/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
        


Could not download Pipeline from pyannote/speaker-diarization-community-1.
It might be because the repository is private or gated:

* visit https://hf.co/pyannote/speaker-diarization-community-1 to accept user conditions
* visit https://hf.co/settings/tokens to create an authentication token
* load the Pipeline with the `token` argument:
    >>> Pipeline.from_pretrained('pyannote/speaker-diarization-community-1', token='hf_....')



GatedRepoError: 401 Client Error. (Request ID: Root=1-69b04ef4-1df66b49649e2a0a57d1b887;196c27ee-20ef-41b5-a390-3e6f04ee407f)

Cannot access gated repo for url https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/config.yaml.
Access to model pyannote/speaker-diarization-community-1 is restricted. You must have access to it and be authenticated to access it. Please log in.